# Catalog builder

Turn a parent galaxy catalog (COSMOS photometry, a detection-truth catalog, …)
into the **train / eval FITS catalogs** ShearNet samples morphologies and shears
from.

**Pipeline:** load & standardise → convert (q, φ) → (g1, g2) → filter (HLR, flux)
→ optional rotation augmentation → train/eval split → write FITS → validate
(KS test + distribution plots).

The output FITS carries the columns `generate_dataset` reads — `G1, G2, HLR,
FLUX` (plus `Q, PHI` for provenance). Point `catalog.cosmos_cat_fname` in a config
(or `--cosmos_cat_fname`) at the **training** FITS to train on real morphologies
instead of the synthetic fallback.

Only needs `numpy`, `astropy`, `galsim`, `scipy`, `matplotlib` — no model or GPU.

In [ ]:
import os
import numpy as np
import galsim
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import fits
from scipy.stats import ks_2samp

## 1 · Load & standardise

**Edit this cell** for your catalog. Map its columns onto four raw arrays: axis
ratio `q_raw`, position angle `phi_raw` [rad], half-light radius `hlr_raw`
[arcsec], and `flux_raw` [counts]. Two worked examples are shown — keep one.

In [ ]:
# ---- Option A: COSMOS photometry CSV ------------------------------------------
CATALOG = "/path/to/cosmos15_superbit2023_phot_shapes_with_sigma.csv"
cat = Table.read(CATALOG, format="csv")
q_raw   = np.asarray(cat["c10_sersic_fit_q"],   dtype=np.float64)
phi_raw = np.asarray(cat["c10_sersic_fit_phi"], dtype=np.float64)                 # radians
hlr_raw = np.asarray(cat["c10_sersic_fit_hlr"], dtype=np.float64) * 0.03 * np.sqrt(q_raw)
hlr_raw = np.minimum(hlr_raw, 1.0)                                               # cap at 1 arcsec
flux_raw = np.asarray(cat["crates_b"], dtype=np.float64) * 300 * 36 / 0.343

# ---- Option B: detection-truth FITS (uncomment to use) ------------------------
# CATALOG = "/path/to/simulated_detection_truth.fits"
# cat = Table.read(CATALOG)
# q_raw    = np.asarray(cat["c10_sersic_fit_q"],   dtype=np.float64)
# phi_raw  = np.asarray(cat["c10_sersic_fit_phi"], dtype=np.float64)
# hlr_raw  = np.asarray(cat["c10_sersic_fit_hlr"], dtype=np.float64)
# flux_raw = np.asarray(cat["flux"], dtype=np.float64)

OUT_DIR = "."          # directory for the output FITS files
OUT_TAG = "COSMOS"     # suffix used in the output filenames
print(f"Loaded {len(q_raw)} rows from {os.path.basename(CATALOG)}")

## 2 · Convert (q, φ) → reduced shear (g1, g2)

Using GalSim's shear convention, with a `q > 1` guard (swap major/minor axis).

In [ ]:
def qphi_to_g1g2(q, phi):
    q = np.asarray(q, dtype=np.float64)
    phi = np.asarray(phi, dtype=np.float64)
    g1 = np.empty(q.shape, dtype=np.float64)
    g2 = np.empty(q.shape, dtype=np.float64)
    for i in range(q.size):
        qi = q[i]
        if qi > 1.0:
            qi = 1.0 / qi
        s = galsim.Shear(q=float(qi), beta=float(phi[i]) * galsim.radians)
        g1[i], g2[i] = s.g1, s.g2
    return g1, g2

g1_raw, g2_raw = qphi_to_g1g2(q_raw, phi_raw)
print(f"g1 range [{g1_raw.min():+.3f}, {g1_raw.max():+.3f}], "
      f"g2 range [{g2_raw.min():+.3f}, {g2_raw.max():+.3f}]")

## 3 · Filter

Drop non-finite / non-positive HLR or flux, then clip flux outliers in log space
(σ-clip or IQR). One shared mask is applied to every array, so row *i* always
refers to the same galaxy.

In [ ]:
FLUX_METHOD = "sigma"   # "sigma" (mean ± N·std of log10 flux) or "iqr"
FLUX_NSIGMA = 3.0

valid = (np.isfinite(hlr_raw) & (hlr_raw > 1e-6)
         & np.isfinite(flux_raw) & (flux_raw > 0))

logf = np.full(flux_raw.shape, np.nan)
logf[flux_raw > 0] = np.log10(flux_raw[flux_raw > 0])

if FLUX_METHOD == "sigma":
    mu, sd = np.nanmean(logf), np.nanstd(logf)
    lo, hi = mu - FLUX_NSIGMA * sd, mu + FLUX_NSIGMA * sd
elif FLUX_METHOD == "iqr":
    q1, q3 = np.nanpercentile(logf, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
else:
    raise ValueError("FLUX_METHOD must be 'sigma' or 'iqr'")

valid &= (logf >= lo) & (logf <= hi)

q_arr, phi_arr = q_raw[valid], phi_raw[valid]
g1_arr, g2_arr = g1_raw[valid], g2_raw[valid]
hlr_arr, flux_arr = hlr_raw[valid], flux_raw[valid]
print(f"Kept {int(valid.sum())} / {valid.size} galaxies "
      f"(log10 flux in [{lo:.2f}, {hi:.2f}])")

## 4 · Optional rotation augmentation

Grow the catalog by re-drawing the position angle uniformly (`q`, `hlr`, `flux`
are tiled, `g1`/`g2` recomputed). `N_AUGMENTS = 1` disables it.

In [ ]:
N_AUGMENTS = 1     # 1 = off; >1 = this many random rotations per galaxy
AUG_SEED = 42

if N_AUGMENTS > 1:
    rng = np.random.default_rng(AUG_SEED)
    q_arr    = np.tile(q_arr, N_AUGMENTS)
    hlr_arr  = np.tile(hlr_arr, N_AUGMENTS)
    flux_arr = np.tile(flux_arr, N_AUGMENTS)
    phi_arr  = rng.uniform(0.0, 2 * np.pi, size=q_arr.size)
    g1_arr, g2_arr = qphi_to_g1g2(q_arr, phi_arr)
    print(f"Augmented to {q_arr.size} galaxies ({N_AUGMENTS}x)")
else:
    print(f"No augmentation ({q_arr.size} galaxies)")

## 5 · Train / eval split

In [ ]:
RANDOM_SEED = 42
TRAIN_FRAC = 3 / 5

rng = np.random.default_rng(RANDOM_SEED)
n_total = q_arr.size
perm = rng.permutation(n_total)
n_train = int(round(n_total * TRAIN_FRAC))
train_idx, eval_idx = perm[:n_train], perm[n_train:]
print(f"train: {train_idx.size}   eval: {eval_idx.size}")

## 6 · Write FITS

`G1, G2, HLR, FLUX` are the columns `generate_dataset` reads; `Q, PHI` are kept
for provenance.

In [ ]:
def make_fits(idx, subset):
    cols = [
        fits.Column(name="Q",    format="D", array=q_arr[idx]),
        fits.Column(name="PHI",  format="D", array=phi_arr[idx], unit="rad"),
        fits.Column(name="G1",   format="D", array=g1_arr[idx]),
        fits.Column(name="G2",   format="D", array=g2_arr[idx]),
        fits.Column(name="HLR",  format="D", array=hlr_arr[idx], unit="arcsec"),
        fits.Column(name="FLUX", format="D", array=flux_arr[idx], unit="count"),
    ]
    hdu = fits.BinTableHDU.from_columns(cols)
    hdu.header["SUBSET"] = (subset, "train or eval subset")
    hdu.header["RNDSD"]  = (RANDOM_SEED, "split RNG seed")
    hdu.header["NGAL"]   = (len(idx), "galaxies in this subset")
    hdu.header["ORIGIN"] = (os.path.basename(CATALOG), "source catalog")
    return hdu

os.makedirs(OUT_DIR, exist_ok=True)
train_path = os.path.join(OUT_DIR, f"cosmos_catalog_train_{OUT_TAG}.fits")
eval_path  = os.path.join(OUT_DIR, f"cosmos_catalog_eval_{OUT_TAG}.fits")
make_fits(train_idx, "TRAIN").writeto(train_path, overwrite=True)
make_fits(eval_idx,  "EVAL").writeto(eval_path,  overwrite=True)
print("wrote", train_path)
print("wrote", eval_path)

## 7 · Validate the split

A two-sample KS test per parameter (train vs eval). Large p-values mean the two
subsets look like samples of the same distribution — exactly what we want.

In [ ]:
params = [
    ("Q",    q_arr,    r"axis ratio $q$"),
    ("PHI",  phi_arr,  r"position angle $\phi$ [rad]"),
    ("G1",   g1_arr,   r"$g_1$"),
    ("G2",   g2_arr,   r"$g_2$"),
    ("HLR",  hlr_arr,  r"HLR [arcsec]"),
    ("FLUX", flux_arr, r"flux [count]"),
]
print(f"{'param':<6}{'KS stat':>12}{'p-value':>12}   same dist (p>0.05)?")
print("-" * 48)
for name, arr, _ in params:
    stat, p = ks_2samp(arr[train_idx], arr[eval_idx])
    print(f"{name:<6}{stat:>12.4f}{p:>12.4f}   {'yes' if p > 0.05 else 'NO  <-- check'}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, arr, label) in zip(axes.flat, params):
    use_log = name == "FLUX"
    tr = np.log10(arr[train_idx]) if use_log else arr[train_idx]
    ev = np.log10(arr[eval_idx]) if use_log else arr[eval_idx]
    bins = np.linspace(min(tr.min(), ev.min()), max(tr.max(), ev.max()), 40)
    ax.hist(tr, bins=bins, density=True, alpha=0.5, label="train")
    ax.hist(ev, bins=bins, density=True, alpha=0.5, label="eval")
    ax.set_xlabel(("log10 " if use_log else "") + label)
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("Train vs eval parameter distributions", fontsize=13)
plt.tight_layout()
plt.show()